In [1]:
# =========================================
# 04_merge_master_table.ipynb
# Merge labels + architecture + stress into one leakage-safe master table
# =========================================

import os
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 200)

WORK_DIR = r"C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026"

LABEL_FILE = os.path.join(WORK_DIR, "interim", "t80_labels.parquet")
ARCH_FILE = os.path.join(WORK_DIR, "interim", "architecture_features.parquet")
STRESS_FILE = os.path.join(WORK_DIR, "interim", "stress_features.parquet")

INTERIM_DIR = os.path.join(WORK_DIR, "interim")
META_DIR = os.path.join(WORK_DIR, "metadata")
REPORT_DIR = os.path.join(WORK_DIR, "reports")
ARTIFACT_DIR = os.path.join(WORK_DIR, "artifacts", "04_merge")

for p in [INTERIM_DIR, META_DIR, REPORT_DIR, ARTIFACT_DIR]:
    os.makedirs(p, exist_ok=True)

print("LABEL_FILE:", LABEL_FILE)
print("ARCH_FILE:", ARCH_FILE)
print("STRESS_FILE:", STRESS_FILE)

LABEL_FILE: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\t80_labels.parquet
ARCH_FILE: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\architecture_features.parquet
STRESS_FILE: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\stress_features.parquet


In [2]:
labels = pd.read_parquet(LABEL_FILE)
arch = pd.read_parquet(ARCH_FILE)
stress = pd.read_parquet(STRESS_FILE)

print("labels shape :", labels.shape)
print("arch shape   :", arch.shape)
print("stress shape :", stress.shape)

labels shape : (1835, 11)
arch shape   : (1835, 36)
stress shape : (1835, 27)


In [3]:
for name, df_ in [("labels", labels), ("arch", arch), ("stress", stress)]:
    if "raw_row_id" not in df_.columns:
        raise ValueError(f"{name} missing raw_row_id")
    if df_["raw_row_id"].duplicated().any():
        dup_n = df_["raw_row_id"].duplicated().sum()
        raise ValueError(f"{name} has duplicated raw_row_id values: {dup_n}")

print("All tables have unique raw_row_id.")

All tables have unique raw_row_id.


In [4]:
arch_drop = [c for c in ["Ref_DOI_number", "Cell_architecture"] if c in arch.columns]
stress_drop = [c for c in ["Ref_DOI_number"] if c in stress.columns]

arch_m = arch.drop(columns=arch_drop, errors="ignore").copy()
stress_m = stress.drop(columns=stress_drop, errors="ignore").copy()

print("arch merge shape   :", arch_m.shape)
print("stress merge shape :", stress_m.shape)

arch merge shape   : (1835, 34)
stress merge shape : (1835, 26)


In [5]:
master = labels.merge(
    arch_m,
    on="raw_row_id",
    how="left",
    validate="one_to_one"
)

print("After labels + arch:", master.shape)
master.head(2)

After labels + arch: (1835, 44)


,raw_row_id,Ref_DOI_number,Ref_publication_date,publication_year,Cell_architecture,Stability_protocol,Encapsulation,Stability_PCE_T80,T80_raw,T80_clean,T80_log1p,architecture_family,is_nip,is_pin,is_other_arch,etl_family,htl_family,backcontact_family,etl_has_tio2,etl_has_sno2,etl_has_pcbm,etl_has_c60,etl_has_zno,htl_has_spiro,htl_has_ptaa,htl_has_pedot,htl_has_niox,htl_has_p3ht,back_has_au,back_has_ag,back_has_al,back_has_carbon,has_perovskite_additives,has_etl_additives,has_htl_additives,band_gap_ev,perovskite_thickness_nm,etl_thickness_nm,cell_area_measured_cm2,n_cells_per_substrate,band_gap_ev_missing,perovskite_thickness_nm_missing,etl_thickness_nm_missing,cell_area_measured_cm2_missing
0,27,10.1039/c9ta01893j,20/03/2019,2019,nip,ISOS-L-1,False,200.0,200.0,200.0,5.303305,nip,1,0,0,tio2,spiro_ometad,au,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,1.59,NaN,NaN,0.06,0.0,0,1,1,0
1,45,10.1002/aenm.201803587,06/03/2019,2019,nip,ISOS-L-1,False,150.0,150.0,150.0,5.017280,nip,1,0,0,sno2,spiro_ometad,au,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,NaN,500.0,25.0,0.16,0.0,1,0,0,0


In [6]:
master = master.merge(
    stress_m,
    on="raw_row_id",
    how="left",
    validate="one_to_one"
)

print("After + stress:", master.shape)
master.head(2)

After + stress: (1835, 69)


,raw_row_id,Ref_DOI_number,Ref_publication_date,publication_year,Cell_architecture,Stability_protocol,Encapsulation,Stability_PCE_T80,T80_raw,T80_clean,T80_log1p,architecture_family,is_nip,is_pin,is_other_arch,etl_family,htl_family,backcontact_family,etl_has_tio2,etl_has_sno2,etl_has_pcbm,etl_has_c60,etl_has_zno,htl_has_spiro,htl_has_ptaa,htl_has_pedot,htl_has_niox,htl_has_p3ht,back_has_au,back_has_ag,back_has_al,back_has_carbon,has_perovskite_additives,has_etl_additives,has_htl_additives,band_gap_ev,perovskite_thickness_nm,etl_thickness_nm,cell_area_measured_cm2,n_cells_per_substrate,band_gap_ev_missing,perovskite_thickness_nm_missing,etl_thickness_nm_missing,cell_area_measured_cm2_missing,encapsulation_flag,encapsulation_missing,protocol_family,protocol_has_l,protocol_has_d,protocol_has_o,bias_family,bias_is_mpp,bias_is_oc,bias_is_sc,bias_mentions_dark,light_intensity_suns,light_intensity_missing,light_bin,is_dark_condition,is_approx_1sun,is_high_light,temperature_c,temperature_missing,rh_pct,rh_missing,is_room_temperature,is_hot_test,is_dry_condition,is_humid_condition
0,27,10.1039/c9ta01893j,20/03/2019,2019,nip,ISOS-L-1,False,200.0,200.0,200.0,5.303305,nip,1,0,0,tio2,spiro_ometad,au,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,1.59,NaN,NaN,0.06,0.0,0,1,1,0,0.0,0,isos_l,1,0,0,mpp,1,0,0,0,100.0,0,high_light,0,0,1,25.0,0,NaN,1,1,0,0,0
1,45,10.1002/aenm.201803587,06/03/2019,2019,nip,ISOS-L-1,False,150.0,150.0,150.0,5.017280,nip,1,0,0,sno2,spiro_ometad,au,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,NaN,500.0,25.0,0.16,0.0,1,0,0,0,0.0,0,isos_l,1,0,0,mpp,1,0,0,0,100.0,0,high_light,0,0,1,25.0,0,NaN,1,1,0,0,0


In [7]:
print("Rows:", len(master))
print("Unique raw_row_id:", master["raw_row_id"].nunique())
print("Unique DOI groups:", master["Ref_DOI_number"].nunique())
print("Missing T80_clean:", master["T80_clean"].isna().sum())
print("Missing T80_log1p:", master["T80_log1p"].isna().sum())

Rows: 1835
Unique raw_row_id: 1835
Unique DOI groups: 964
Missing T80_clean: 0
Missing T80_log1p: 0


In [8]:
forbidden_exact = [
    "Stability_PCE_T80",
    "T80_raw",
    "T80_clean",
    "T80_log1p",
]

# these are kept in master for reference, but not to be used as model features later
group_and_meta_cols = [
    "raw_row_id",
    "Ref_DOI_number",
    "Ref_publication_date",
    "publication_year",
]

print("Forbidden target columns present:")
print([c for c in forbidden_exact if c in master.columns])

print("\nMetadata/group columns present:")
print([c for c in group_and_meta_cols if c in master.columns])

Forbidden target columns present:
['Stability_PCE_T80', 'T80_raw', 'T80_clean', 'T80_log1p']

Metadata/group columns present:
['raw_row_id', 'Ref_DOI_number', 'Ref_publication_date', 'publication_year']


In [9]:
missing_df = pd.DataFrame({
    "column": master.columns,
    "missing_pct": master.isna().mean().values * 100,
    "dtype": master.dtypes.astype(str).values
}).sort_values(["missing_pct", "column"])

missing_df.to_csv(os.path.join(REPORT_DIR, "04_master_missingness.csv"), index=False)
missing_df.head(40)

,column,missing_pct,dtype
4,Cell_architecture,0.0,str
6,Encapsulation,0.0,str
1,Ref_DOI_number,0.0,str
2,Ref_publication_date,0.0,str
7,Stability_PCE_T80,0.0,float64
5,Stability_protocol,0.0,str
9,T80_clean,0.0,float64
10,T80_log1p,0.0,float64
8,T80_raw,0.0,float64
11,architecture_family,0.0,str


In [10]:
constant_rows = []

for c in master.columns:
    nunq = master[c].nunique(dropna=True)
    constant_rows.append({
        "column": c,
        "n_unique_non_missing": int(nunq),
        "is_constant_or_empty": bool(nunq <= 1)
    })

constant_df = pd.DataFrame(constant_rows).sort_values(["is_constant_or_empty", "column"], ascending=[False, True])
constant_df.to_csv(os.path.join(REPORT_DIR, "04_master_constant_columns.csv"), index=False)

constant_df.head(40)

,column,n_unique_non_missing,is_constant_or_empty
54,bias_mentions_dark,1,True
45,encapsulation_missing,1,True
49,protocol_has_o,1,True
4,Cell_architecture,4,False
6,Encapsulation,2,False
1,Ref_DOI_number,964,False
2,Ref_publication_date,669,False
7,Stability_PCE_T80,492,False
5,Stability_protocol,16,False
9,T80_clean,492,False


In [11]:
dup_cols = pd.Series(master.columns).value_counts()
dup_cols = dup_cols[dup_cols > 1]

print("Duplicated column names:")
print(dup_cols if len(dup_cols) else "None")

Duplicated column names:
None


In [12]:
label_cols = [
    "raw_row_id",
    "Ref_DOI_number",
    "Ref_publication_date",
    "publication_year",
    "Cell_architecture",
    "Stability_protocol",
    "Encapsulation",
    "Stability_PCE_T80",
    "T80_raw",
    "T80_clean",
    "T80_log1p",
]

label_cols = [c for c in label_cols if c in master.columns]
other_cols = [c for c in master.columns if c not in label_cols]

master = master[label_cols + other_cols].copy()

print("Reordered master shape:", master.shape)

Reordered master shape: (1835, 69)


In [13]:
target_cols = [c for c in ["T80_clean", "T80_log1p"] if c in master.columns]
group_cols = [c for c in ["Ref_DOI_number"] if c in master.columns]
id_cols = [c for c in ["raw_row_id"] if c in master.columns]

metadata_cols = [c for c in ["Ref_publication_date", "publication_year"] if c in master.columns]

preview_exclude_from_model = target_cols + group_cols + id_cols + metadata_cols

pd.DataFrame({
    "exclude_from_model_preview": preview_exclude_from_model
}).to_csv(os.path.join(REPORT_DIR, "04_exclude_from_model_preview.csv"), index=False)

preview_exclude_from_model

['T80_clean',
 'T80_log1p',
 'Ref_DOI_number',
 'raw_row_id',
 'Ref_publication_date',
 'publication_year']

In [14]:
master_parquet = os.path.join(INTERIM_DIR, "master_table.parquet")
master_csv = os.path.join(INTERIM_DIR, "master_table.csv")

master.to_parquet(master_parquet, index=False)
master.to_csv(master_csv, index=False)

print("Saved:", master_parquet)
print("Saved:", master_csv)

Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\master_table.parquet
Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\master_table.csv


In [15]:
model_parquet = os.path.join(INTERIM_DIR, "model_table_T80_broad.parquet")
model_csv = os.path.join(INTERIM_DIR, "model_table_T80_broad.csv")

master.to_parquet(model_parquet, index=False)
master.to_csv(model_csv, index=False)

print("Saved:", model_parquet)
print("Saved:", model_csv)

Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\model_table_T80_broad.parquet
Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\model_table_T80_broad.csv


In [16]:
merge_card = {
    "notebook": "04_merge_master_table.ipynb",
    "inputs": {
        "labels": LABEL_FILE,
        "architecture": ARCH_FILE,
        "stress": STRESS_FILE,
    },
    "row_counts": {
        "labels_rows": int(len(labels)),
        "arch_rows": int(len(arch)),
        "stress_rows": int(len(stress)),
        "master_rows": int(len(master)),
    },
    "group_summary": {
        "unique_doi_groups": int(master["Ref_DOI_number"].nunique()),
        "missing_doi": int(master["Ref_DOI_number"].isna().sum()),
    },
    "target_summary": {
        "missing_T80_clean": int(master["T80_clean"].isna().sum()),
        "missing_T80_log1p": int(master["T80_log1p"].isna().sum()),
    },
    "notes": [
        "Master table retains labels, grouping columns, and engineered features.",
        "Target and metadata columns are still present here for reference.",
        "Final feature exclusion for modeling will be enforced in notebook 05.",
        "Constant and fully missing columns are audited here, not yet removed."
    ],
    "saved_files": {
        "master_parquet": master_parquet,
        "master_csv": master_csv,
        "model_parquet": model_parquet,
        "model_csv": model_csv,
    }
}

merge_card_path = os.path.join(META_DIR, "merge_card.json")

with open(merge_card_path, "w", encoding="utf-8") as f:
    json.dump(merge_card, f, indent=4)

print("Saved:", merge_card_path)

Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\metadata\merge_card.json


In [17]:
print("========== MASTER MERGE SUMMARY ==========")
print("Labels rows:", len(labels))
print("Architecture rows:", len(arch))
print("Stress rows:", len(stress))
print("Master rows:", len(master))
print("Unique DOI groups:", master["Ref_DOI_number"].nunique())
print("Missing DOI rows:", master["Ref_DOI_number"].isna().sum())
print("Missing T80_clean:", master["T80_clean"].isna().sum())
print("Missing T80_log1p:", master["T80_log1p"].isna().sum())
print("Valid temperature:", master["temperature_c"].notna().sum())
print("Valid RH:", master["rh_pct"].notna().sum())

print("\nColumns with 100% missing:")
print(missing_df.loc[missing_df["missing_pct"] == 100, "column"].tolist())

print("\nConstant or empty columns:")
print(constant_df.loc[constant_df["is_constant_or_empty"], "column"].tolist())

print("\nSaved merged tables successfully.")

========== MASTER MERGE SUMMARY ==========
Labels rows: 1835
Architecture rows: 1835
Stress rows: 1835
Master rows: 1835
Unique DOI groups: 964
Missing DOI rows: 0
Missing T80_clean: 0
Missing T80_log1p: 0
Valid temperature: 1768
Valid RH: 1343

Columns with 100% missing:
[]

Constant or empty columns:
['bias_mentions_dark', 'encapsulation_missing', 'protocol_has_o']

Saved merged tables successfully.
